# Compiling Sundown County Newspaper Data

The following notebook outlines how I compiled sundown county newspaper data. Within the context of my work, sundown counties and towns are defined not exclusively by municipal or jurisdictional boundaries. Rather, they are places where known threats of racial violence existed in American history–threats known because they were enacted, culminating in lynchings that were subsequently documented by newspapers and data records. Compiling this data therefore relies on two major resources: the digitized newspapers in the Chronicling America archive and lynching records compiled by multiple, centuries' long efforts.

My process does not technically begin here. I've previously downloaded the entirety of the OCR data in Chronicling America (as of June 2025). To review those steps, check out this blog post: https://matthewkollmer.com/how-im-backing-up-chronicling-america/, and my GitHub repo here: https://github.com/MatthewKollmer/chron_am_backup/tree/main. These prior steps describe some dataframes used in this notebook as well: batches_df and papers_df. These dataframes contain information about how my local OCR version of Chron Am is stored and organized. If you're following along, you'll have an easier time if you replicate these local Chron Am steps, too. That's not to say it's easy or fast to do so–not at all, downloading the entirety of Chron Am's OCR is time-consuming and requires several TB of storage space. But it does make the following project a lot faster and easier to follow.

My process also utilizes Chron Am's list of newspapers which can be downloaded here: https://www.loc.gov/collections/chronicling-america/titles/. It is appears in the process as the dataframe called 'newspapers'. Quite simply, it contains information about all the newspapers in Chronicling America.

You'll also notice a dataframe called 'lynchings'. This is a combined inventory of of two lynching databases–a concatenation of the Tolnay-Beck-Bailey inventory and the Seguin-Rigby Dataset. There's a lot of information available online about these datasets, so I'll leave it to you to do explore supplementary readings and research. As a starting point, though, my previous project, the Dataset of U.S. Lynching Reports (DUSLR), may prove useful. See here for its GitHub Repo: https://github.com/MatthewKollmer/DUSLR. The first Python notebook in that repository (01_unify_data_sources.ipynb), describes how I concatenated these two data sources into one file.

To the process at hand: I began as always by loading the necessary Python libraries and dataframes:

In [ ]:
import pandas as pd
import ast
import os
import tarfile
import shutil
from pathlib import Path

newspapers = pd.read_csv('chron_am_newspapers.csv')
lynchings = pd.read_csv('combined_inventories.csv')
batches_df = pd.read_csv('https://raw.githubusercontent.com/MatthewKollmer/chron_am_backup/refs/heads/main/ocr_batches.csv', converters={'contents': ast.literal_eval}) # converters here ensures the lists of dictionaries in ocr_batches.csv are read as such
papers_df = pd.read_csv('https://raw.githubusercontent.com/MatthewKollmer/chron_am_backup/refs/heads/main/newspapers.csv')

Ope–almost forgot: in full disclosure, the majority of this code was written with Claude Code and/or Codex. I review and revise outputs, but it's safe to say that over half of the lines were directly taken from LLMs. This is just a developing reality about my programming process. These coding tools have proven to make the writing of code go much faster. I still have to conceive of the whole process, but these coding LLMs can turn my pseudocode into actual Python a lot faster than I can.

Anyway, I started by preprocessing some of the newspaper data in order to make it more seamlessly compatible with downstream processes. This included standardizing state names to abbreviations:

In [ ]:
state_map = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN',
    'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT',
    'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD',
    'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
    'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY',
}

newspapers['State'] = (
    newspapers['State']
    .str.strip()
    .str.title()
    .map(state_map)
)

And then standardizing county names by removing the words 'county' and 'parish'.

In [ ]:
newspapers['County'] = (
    newspapers['County']
    .str.lower()
    .str.replace(r'\b(county|parish)\b', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

I did the same with county data in the lynchings dataframe. I also made the decision to change day values in lynchings from 0 (if that's how they were entered) to 1. Basically, the curators of the lynching data entered 0 if they didn't know the precise date of the lynching. This would be fine except it makes the computational parsing of the date impossible. So, since I didn't need to know the precise date, just a general day within the month, I chose to turn 0 values to 1. Then I parsed the dates in lynchings to follow a standard yyyy-mm-dd format.

In [ ]:
lynchings['county'] = (
    lynchings['county']
    .str.lower()
    .str.replace(r'\b(county|parish)\b', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

day_fixed = pd.to_numeric(lynchings['day'], errors='coerce')
day_fixed = day_fixed.replace(0, 1)

lynchings['lynch_date'] = pd.to_datetime(
    {'year': lynchings['year'], 'month': lynchings['month'], 'day': day_fixed},
    errors='coerce'
).dt.strftime('%Y-%m-%d')

Next I merged the newspapers data with the lynchings data by state and county. This resulted in a new dataframe called matched which conveniently paired lynchings and newspapers by their shared geographic locations (state and county).

In [ ]:
matched = newspapers.merge(
    lynchings,
    left_on=['State', 'County'],
    right_on=['state', 'county'],
    how='inner',
)

But I also needed to ensure the newspapers contained digitized OCR data corresponding to the period of the lynching–a temporal match as well. To do this, I standardized the date formatting of the First Issue, Last Issue, and lynch_date columns. Then I narrowed the data to just cases where the lynching date landed between the first issue and last issue of the corresponding newspapers.

In [ ]:
matched['First Issue'] = pd.to_datetime(matched['First Issue'], errors='coerce')
matched['Last Issue'] = pd.to_datetime(matched['Last Issue'], errors='coerce')
matched['lynch_date'] = pd.to_datetime(matched['lynch_date'], errors='coerce')

matched['lynch_in_range'] = matched['lynch_date'].between(
    matched['First Issue'], matched['Last Issue'], inclusive='both'
)

matched = matched[matched['lynch_in_range']]

Next I looked to my local Chron Am data. I needed to merge the matched dataframe with the dataframes that provided information about the structure and contents of my local Chron Am files (namely papers_df). To do this, I merged them on their corresponding LCCN values. These LCCN values are Chron Am's unique identifiers for every newspaper in their archive.

In [ ]:
matched = matched.merge(
    papers_df[['LCCN', 'tarfiles']],
    on='LCCN',
    how='left'
)

At this point, my matched dataframe provided a pretty solid amount of information. It contained a narrowed list of newspapers and lynchings where they corresponded temporally and geographically by county. It also contained matched LCCN values and their corresponding batch files in my local Chron Am directory. But one complicating factor remained. In my local Chron Am directory, newspapers were often split across multiple batch files. This meant I also needed to know precisely what years I wanted to draw from and which years were in which batch files for every relevant newspaper.

This required that I specify a temporal window. I chose the four years before a lynching occurred, the year of its occurrence, and three years afterwards. I chose this eight-year window because it seemed to me to be a fair definition of "contemporary" to the lynching events within these places. It was also a guess that it would yield relatively manageable amounts of data. I didn't want so much data that running further analyses and tasks would be too difficult or time-consuming. I also wanted enough contemporary newspaper coverage to be worth analyzing. Based on the results, I think I guessed well (see below). But this choice is worth noting because there could be far wider temporal windows in these analyses. I'd be interested in decades-long changes to sundown town or county coverage as well, but that's for further work.

To capture these temporal windows, I used the following functions, the first turning the tarfiles column values into lists (rather than dictionaries), and the second identifying lynching years in these lists and saving their corresponding batch file names in a new column called batches_to_pull.

In [ ]:
def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

matched['tarfiles'] = matched['tarfiles'].map(to_list)

matched['lynch_year'] = pd.to_datetime(matched['lynch_date'], errors='coerce').dt.year

def pick_batches(tarfiles, lynch_year):
    if not tarfiles or pd.isna(lynch_year):
        return []
    window = set(range(int(lynch_year) - 4, int(lynch_year) + 3))
    out = []
    for d in tarfiles:
        years = [y for y in d.get('years', []) if y in window]
        if years:
            out.append({'file_name': d.get('file_name'), 'years': years})
    return out

matched['batches_to_pull'] = matched.apply(
    lambda r: pick_batches(r['tarfiles'], r['lynch_year']),
    axis=1
)

At this point, I also chose to remove cases where victims were white. This is because I only wanted cases where the catalyst for lynchings was racism–where it reflected a heightened degree of racial threat within the community and could therefore be used to study the phenomenological and epistemological dimensions of sundown towns. While many white victims of lynchings were targeted for their support for civil rights and social justice, this filtering out of their cases was also a straightforward way to narrow my data to potential sundown towns.

In [ ]:
matched = matched[~matched['race'].str.contains('White', case=False, na=False)]

With these preprocessing steps in place, I was ready to send my computer to task, combing through my local Chron Am directory and extracting only the OCR text from the newspapers that:

1) were from counties where lynchings occurred, and
2) had digitized newspaper pages within the eight-year window around the lynchings

To do this, I established by file paths (TAR_ROOT and OUT_ROOT), set a limit to the amount of data the code would compile (no more than 500GB, to ensure I had space on my device to store all of it), and combed the tarball files without unpacking them since that would expend my storage space as well. For any relevant files in the tarballs, this code created a copy unpacked and separated from the rest of the Chron Am data–a subset of only OCR newspaper text from counties where lynchings occurred.

In [ ]:
TAR_ROOT = '/Volumes/t5_evo_8tb/ChronAm/tarbiz2_files'
OUT_ROOT = '/Volumes/t5_evo_8tb/ChronAm/sundown_town_data'

MAX_BYTES = 500 * 1024**3  # this is just to ensure I don't come close to overfilling my SSD (set to 500GB)
total_written = 0

class StopCopy(Exception):
    pass

tar_to_targets = {}

for _, row in matched.iterrows():
    lccn = row['LCCN']
    victim = str(row['victim']).strip()
    batches = row.get('batches_to_pull', []) or []
    for b in batches:
        tar_name = b.get('file_name')
        years = b.get('years', [])
        if not tar_name or not years:
            continue
        tar_to_targets.setdefault(tar_name, []).append((lccn, set(map(str, years)), victim))

def member_matches(member_name, lccn, years_set):
    parts = member_name.split('/')
    if lccn not in parts:
        return False, None
    for y in years_set:
        if y in parts:
            return True, parts
    return False, None

try:
    for tar_name, targets in tar_to_targets.items():
        tar_path = os.path.join(TAR_ROOT, tar_name)
        if not os.path.exists(tar_path):
            print(f'Missing tarball: {tar_path}')
            continue

        with tarfile.open(tar_path, 'r:bz2') as tf:
            for member in tf:
                if not member.isfile() or not member.name.endswith('.txt'):
                    continue

                for lccn, years_set, victim in targets:
                    ok, parts = member_matches(member.name, lccn, years_set)
                    if not ok:
                        continue

                    lccn_idx = parts.index(lccn)
                    rel_parts = parts[lccn_idx:]
                    dest_path = os.path.join(OUT_ROOT, victim, *rel_parts)

                    if os.path.exists(dest_path):
                        break

                    # Stop if this file would push us over the 500GB cap
                    if total_written + member.size > MAX_BYTES:
                        raise StopCopy(
                            f'Stopping: {total_written/1024**3:.2f}GB written, '
                            f'next file is {member.size/1024**3:.2f}GB.'
                        )

                    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

                    f = tf.extractfile(member)
                    if f is None:
                        break
                    with f, open(dest_path, 'wb') as out_f:
                        shutil.copyfileobj(f, out_f, length=1024 * 1024)

                    total_written += member.size
                    break

except StopCopy as e:
    print(e)

After about ten hours of runtime, I looked over the results, and I did in fact have a bunch of newspaper data. It was structured in the following way:

- a subdirectory with the victim's name (the case identifier)
- followed by subdirectories of LCCN values (newspaper identifiers)
- followed by subdirectories for years
- followed by subdirectories for months, dates, editions, and page numbers
- followed by ocr.txt files with the OCR for the given page

So, basically, I had a bunch of nested directories with all the available sundown county newspaper data in Chronicling America. This is exactly what I had hoped for, but to get a sense of the data, I thought I should review it quickly in a few ways. First, I wanted to know how many cases were covered. This was as easy–it just required counting the number of subdirectories for victims. The result: 483 cases with coverage.

In [ ]:
# results/data compiled
root = Path('/Volumes/t5_evo_8tb/ChronAm/sundown_town_data')
count = sum(1 for p in root.iterdir() if p.is_dir())
print(f'Number of cases where we have temporally overlapping digitized pages from the same county: {count}')
# Number of cases where we have temporally overlapping digitized pages from the same county: 483

Second, I wanted to know the size of the data in GB. This could be important for future downstream tasks in terms of runtime or necessary processing power. The result: 40.48 GB of data.

In [ ]:
dir_path = Path('/Volumes/t5_evo_8tb/ChronAm/sundown_town_data')

if not dir_path.exists():
    raise FileNotFoundError(f'Path not found: {dir_path}')

total_bytes = sum(
    (Path(root) / name).stat().st_size
    for root, _, files in os.walk(dir_path)
    for name in files
)

print(f'Directory size: {total_bytes / (1024**3):.2f} GB')
# Directory size: 40.48 GB

I also wanted to know exactly how many newspaper pages were included in my data. I counted them by counting the .txt files in the data. The result: 2,519,912 OCR newspaper pages.

In [ ]:
if not dir_path.exists():
    raise FileNotFoundError(f'Path not found: {dir_path}')

txt_count = sum(
    1
    for root, _, files in os.walk(dir_path)
    for name in files
    if name.lower().endswith('.txt')
)

print(f'Number of extracted newspaper pages: {txt_count}')
# Number of extracted newspaper pages: 2519912

That's it–I had something to work with now for analyses. In my following notebooks, I'll describe some further preprocessing steps and how I've used this data to study sundown towns.